# 07 m9.2_physics Counterfactual Window Ranker

This exploratory notebook implements the first executable version of the CGPT Pro counterfactual-ranker recommendation. The model is named `m9.2_physics`.

**Purpose.** Test whether dense, physics-inspired candidate windows plus a per-day ranking model can recover reverse power flow (RPF) windows that m7/m9 missed, while still decoding each site-day to either no RPF or one contiguous RPF window.

**Isolation.** Everything here stays inside `publication/2_journal_article/notebooks/99_Misc/`. The notebook does not modify the journal config, production helpers, Notebook 2, or main output folders.

**Default run.** `RUN_MODE = "smoke"` by default. Smoke mode trains on a tiny Alpha subset and evaluates a few Beta diagnostic days. Focus mode expands the diagnostic Beta examples for `beta_B`, `beta_D`, and `beta_E`. Full mode is coded as an opt-in path, but is not executed by default because dense candidate windows can generate millions of rows.

**Key modelling idea.** For each candidate window `W`, the notebook compares two same-day counterfactual reconstructions:

- `U_empty = solar_MW + net_load_MW`, as if no sign error exists;
- `U_W = solar_MW - net_load_MW` inside `W`, and `solar_MW + net_load_MW` outside `W`.

A plausible RPF window should make the reconstructed underlying-demand shape smoother, more continuous at boundaries, and more consistent with expected solar/net-load co-movement.


## 1. Imports, Controls, And Paths

This section defines the only intended controls for this misc experiment. Use environment variables for fast reruns from the command line:

- `M92_RUN_MODE=smoke`, `focus`, or `full`;
- `M92_REUSE_CACHE=0` to force candidate-feature recomputation;
- `M92_WRITE_HTML=0` to skip Plotly examples.

Smoke and focus modes intentionally keep the Alpha training sample small. They are for implementation checks and visual diagnosis, not final model validation.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import xgboost as xgb

MODEL_NAME = "m9.2_physics"
RUN_MODE = os.environ.get("M92_RUN_MODE", "smoke").strip().lower()
if RUN_MODE not in {"smoke", "focus", "full"}:
    raise ValueError(f"Unsupported RUN_MODE={RUN_MODE!r}; expected smoke, focus, or full.")

RANDOM_SEED = 9
REUSE_CACHED_FEATURES = os.environ.get("M92_REUSE_CACHE", "1").strip() != "0"
WRITE_HTML = os.environ.get("M92_WRITE_HTML", "1").strip() != "0"
P_MIN = 0.70

WINDOW_START_HOUR = 6
WINDOW_END_HOUR = 18
MIN_DURATION_MINUTES = 30
MAX_DURATION_MINUTES = 8 * 60
TOP_SOLAR_PEAKS = 3
SOLAR_PEAK_MIN_FRAC = 0.25
SOLAR_PEAK_MIN_SEPARATION_MINUTES = 90
FEATURE_VERSION = "lean_v1_counterfactual_2026_06_25"

FOCUS_SITES = ["beta_B", "beta_D", "beta_E"]
FOCUS_DATES = [
    ("beta_B", "2023-10-13"),
    ("beta_B", "2024-03-03"),
    ("beta_B", "2024-03-07"),
    ("beta_B", "2023-12-12"),
    ("beta_D", "2023-10-01"),
    ("beta_D", "2023-10-03"),
    ("beta_D", "2024-03-14"),
    ("beta_D", "2024-04-21"),
    ("beta_E", "2023-10-21"),
    ("beta_E", "2023-10-08"),
    ("beta_E", "2023-10-07"),
    ("beta_E", "2023-10-15"),
]

JCOL = {
    "orange": "#eb932c",
    "dark_blue": "#22303d",
    "grey": "#2F4D67",
    "light_grey": "#5C7D99",
    "light_white": "#ebe3e3",
}

plt.rcParams.update({
    "font.family": "Arial",
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 140,
})


def resolve_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "publication" / "2_journal_article" / "dataset" / "final").exists():
            return candidate
        if candidate.name == "2_journal_article" and (candidate / "dataset" / "final").exists():
            return candidate.parent.parent
    raise FileNotFoundError("Could not resolve PyNRPF repo root from current working directory.")


REPO_ROOT = resolve_repo_root()
ARTICLE_ROOT = REPO_ROOT / "publication" / "2_journal_article"
MISC_DIR = ARTICLE_ROOT / "notebooks" / "99_Misc"
OUTPUT_ROOT = MISC_DIR / "outputs" / "07_m9_2_physics_counterfactual_ranker"
CSV_DIR = OUTPUT_ROOT / "csv"
FIG_DIR = OUTPUT_ROOT / "figures"
HTML_DIR = OUTPUT_ROOT / "html_examples"
CACHE_DIR = OUTPUT_ROOT / "cache"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for folder in [CSV_DIR, FIG_DIR, HTML_DIR, CACHE_DIR, MANIFEST_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Run mode: {RUN_MODE}")
print(f"Article root: {ARTICLE_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")


## 2. Dataset Loading And Day-Level Selection

This cell loads the final Alpha/Beta datasets and selects the site-days used in the current run mode.

- Alpha is always the only training source.
- Beta labels are used only for diagnostics and evaluation.
- Smoke mode keeps only a tiny deterministic Alpha subset.
- Focus mode uses a larger but still bounded Alpha subset plus selected Beta failure examples.
- Full mode selects all Alpha and all Beta site-days, but should only be run deliberately.


In [ ]:
EXPECTED_COLUMNS = [
    "substation_id",
    "date",
    "timestamp",
    "net_load_MW",
    "solar_MW",
    "label_interval",
    "label_day",
]


def naive_timestamp(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce").dt.tz_convert(None)


def load_final_dataset(name: str) -> pd.DataFrame:
    path = ARTICLE_ROOT / "dataset" / "final" / f"dataset_{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    missing = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}")
    df = df[EXPECTED_COLUMNS].copy()
    df["timestamp"] = naive_timestamp(df["timestamp"])
    df["date"] = df["date"].astype(str)
    df["substation_id"] = df["substation_id"].astype(str)
    df["label_interval"] = df["label_interval"].astype(bool)
    df["label_day"] = df["label_day"].astype(bool)
    return df.sort_values(["substation_id", "date", "timestamp"]).reset_index(drop=True)


def day_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = (
        df.groupby(["substation_id", "date"], sort=True)
        .agg(
            label_day=("label_day", "max"),
            rpf_intervals=("label_interval", "sum"),
            solar_peak=("solar_MW", "max"),
            net_peak=("net_load_MW", "max"),
            n_rows=("timestamp", "size"),
        )
        .reset_index()
    )
    out["key"] = list(zip(out["substation_id"], out["date"]))
    return out


def deterministic_sample(summary: pd.DataFrame, n_pos: int, n_neg: int) -> set[tuple[str, str]]:
    pos = summary.loc[summary["label_day"]].sort_values(
        ["solar_peak", "rpf_intervals", "substation_id", "date"],
        ascending=[False, False, True, True],
    )
    neg = summary.loc[~summary["label_day"]].sort_values(
        ["solar_peak", "net_peak", "substation_id", "date"],
        ascending=[False, False, True, True],
    )
    keys = set(pos.head(n_pos)["key"]) | set(neg.head(n_neg)["key"])
    return keys


alpha = load_final_dataset("alpha")
beta = load_final_dataset("beta")
alpha_days = day_summary(alpha)
beta_days = day_summary(beta)

if RUN_MODE == "smoke":
    alpha_trainval_keys = deterministic_sample(alpha_days, n_pos=18, n_neg=18)
    beta_eval_keys = set(FOCUS_DATES[:6])
elif RUN_MODE == "focus":
    # Focus mode is for fast diagnostic iteration, not final validation.
    # Keep the training subset deliberately small so a fresh run finishes in minutes.
    alpha_trainval_keys = deterministic_sample(alpha_days, n_pos=24, n_neg=24)
    focus_keys = set(FOCUS_DATES)
    # Add a few high-solar non-RPF days on the same poor-transfer sites so the null candidate is exercised.
    focus_non = beta_days.loc[
        beta_days["substation_id"].isin(FOCUS_SITES) & ~beta_days["label_day"]
    ].sort_values(["solar_peak", "substation_id", "date"], ascending=[False, True, True])
    beta_eval_keys = focus_keys | set(focus_non.head(9)["key"])
elif RUN_MODE == "full":
    alpha_trainval_keys = set(alpha_days["key"])
    beta_eval_keys = set(beta_days["key"])

print(f"Alpha rows: {len(alpha):,}; Alpha site-days: {len(alpha_days):,}; selected Alpha site-days: {len(alpha_trainval_keys):,}")
print(f"Beta rows: {len(beta):,}; Beta site-days: {len(beta_days):,}; selected Beta site-days: {len(beta_eval_keys):,}")


## 3. Cached Site-Day Representation

The dense candidate generator and counterfactual features use the same site-day arrays many times. This cache stores timestamps, net load, solar, pseudo-load, true labels, and true RPF windows once per selected day. This is the main speed guardrail for smoke/focus iteration.


In [ ]:
@dataclass
class DayRecord:
    dataset: str
    substation_id: str
    date: str
    ts: np.ndarray
    net: np.ndarray
    solar: np.ndarray
    labels: np.ndarray
    true_start: pd.Timestamp | None
    true_end: pd.Timestamp | None
    true_indices: np.ndarray
    daytime_indices: np.ndarray
    daily_solar_peak_idx: int | None
    top_solar_peak_indices: list[int]

    @property
    def key(self) -> tuple[str, str]:
        return (self.substation_id, self.date)

    @property
    def has_rpf(self) -> bool:
        return bool(len(self.true_indices) > 0)


def timestamp_minutes(ts: np.ndarray) -> np.ndarray:
    ser = pd.Series(pd.to_datetime(ts))
    return ser.dt.hour.to_numpy() * 60 + ser.dt.minute.to_numpy()


def select_top_solar_peaks(ts: np.ndarray, solar: np.ndarray, daytime_indices: np.ndarray) -> tuple[int | None, list[int]]:
    if len(daytime_indices) == 0:
        return None, []
    vals = solar[daytime_indices]
    if np.all(np.isnan(vals)):
        return None, []
    local_peak_pos = int(np.nanargmax(vals))
    daily_peak_idx = int(daytime_indices[local_peak_pos])
    daily_peak_value = float(solar[daily_peak_idx])
    if not np.isfinite(daily_peak_value) or daily_peak_value <= 0:
        return daily_peak_idx, [daily_peak_idx]

    threshold = daily_peak_value * SOLAR_PEAK_MIN_FRAC
    candidate_indices = []
    for idx in daytime_indices:
        val = solar[idx]
        if not np.isfinite(val) or val < threshold:
            continue
        left = solar[idx - 1] if idx - 1 >= 0 else -np.inf
        right = solar[idx + 1] if idx + 1 < len(solar) else -np.inf
        if val >= left and val >= right:
            candidate_indices.append(int(idx))
    if daily_peak_idx not in candidate_indices:
        candidate_indices.append(daily_peak_idx)

    candidate_indices = sorted(set(candidate_indices), key=lambda i: (-float(solar[i]), i))
    selected = [daily_peak_idx]
    min_sep = SOLAR_PEAK_MIN_SEPARATION_MINUTES / 15
    for idx in candidate_indices:
        if idx == daily_peak_idx:
            continue
        if all(abs(idx - existing) >= min_sep for existing in selected):
            selected.append(int(idx))
        if len(selected) >= TOP_SOLAR_PEAKS:
            break
    return daily_peak_idx, sorted(set(selected))


def build_day_cache(df: pd.DataFrame, dataset: str, keys: set[tuple[str, str]]) -> dict[tuple[str, str], DayRecord]:
    selected = df.loc[list(zip(df["substation_id"], df["date"]))].copy() if False else df.copy()
    selected["_key"] = list(zip(selected["substation_id"], selected["date"]))
    selected = selected.loc[selected["_key"].isin(keys)].drop(columns="_key")
    cache: dict[tuple[str, str], DayRecord] = {}
    for (site, date), group in selected.groupby(["substation_id", "date"], sort=True):
        g = group.sort_values("timestamp")
        ts = g["timestamp"].to_numpy()
        net = g["net_load_MW"].astype(float).to_numpy()
        solar = g["solar_MW"].astype(float).to_numpy()
        labels = g["label_interval"].astype(bool).to_numpy()
        label_idx = np.flatnonzero(labels)
        true_start = pd.Timestamp(ts[int(label_idx[0])]) if len(label_idx) else None
        true_end = pd.Timestamp(ts[int(label_idx[-1])]) if len(label_idx) else None
        minutes = timestamp_minutes(ts)
        daytime_indices = np.flatnonzero((minutes >= WINDOW_START_HOUR * 60) & (minutes <= WINDOW_END_HOUR * 60))
        daily_peak_idx, top_peaks = select_top_solar_peaks(ts, solar, daytime_indices)
        cache[(str(site), str(date))] = DayRecord(
            dataset=dataset,
            substation_id=str(site),
            date=str(date),
            ts=ts,
            net=net,
            solar=solar,
            labels=labels,
            true_start=true_start,
            true_end=true_end,
            true_indices=label_idx,
            daytime_indices=daytime_indices,
            daily_solar_peak_idx=daily_peak_idx,
            top_solar_peak_indices=top_peaks,
        )
    return cache


alpha_cache = build_day_cache(alpha, "alpha", alpha_trainval_keys)
beta_cache = build_day_cache(beta, "beta", beta_eval_keys)
print(f"Cached Alpha days: {len(alpha_cache):,}; cached Beta days: {len(beta_cache):,}")


## 4. Dense Candidate Windows

Each site-day gets exactly one null/no-RPF candidate. Non-null candidates are bounded to 06:00-18:00, have duration between 30 minutes and 8 hours, and must contain the daily solar peak or one of the top three separated solar peaks. This keeps candidate recall high without blindly considering every possible start/end pair.


In [ ]:
def inclusive_duration_minutes(start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> float:
    return float((pd.Timestamp(end_ts) - pd.Timestamp(start_ts)).total_seconds() / 60.0 + 15.0)


def generate_dense_candidates_for_record(rec: DayRecord) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = [{
        "dataset": rec.dataset,
        "substation_id": rec.substation_id,
        "date": rec.date,
        "candidate_id": 0,
        "is_null": True,
        "candidate_start": pd.NaT,
        "candidate_end": pd.NaT,
        "duration_minutes": 0.0,
        "peak_index": -1,
        "peak_time": pd.NaT,
        "contains_daily_solar_peak": False,
    }]
    if len(rec.daytime_indices) == 0 or not rec.top_solar_peak_indices:
        return rows

    seen: set[tuple[int, int]] = set()
    cid = 1
    daytime = np.array(rec.daytime_indices, dtype=int)
    for peak_idx in rec.top_solar_peak_indices:
        if peak_idx < 0:
            continue
        starts = daytime[daytime <= peak_idx]
        ends = daytime[daytime >= peak_idx]
        for s_idx in starts:
            for e_idx in ends:
                if e_idx < s_idx:
                    continue
                duration = inclusive_duration_minutes(pd.Timestamp(rec.ts[s_idx]), pd.Timestamp(rec.ts[e_idx]))
                if duration < MIN_DURATION_MINUTES or duration > MAX_DURATION_MINUTES:
                    continue
                key = (int(s_idx), int(e_idx))
                if key in seen:
                    continue
                seen.add(key)
                rows.append({
                    "dataset": rec.dataset,
                    "substation_id": rec.substation_id,
                    "date": rec.date,
                    "candidate_id": cid,
                    "is_null": False,
                    "candidate_start": pd.Timestamp(rec.ts[s_idx]),
                    "candidate_end": pd.Timestamp(rec.ts[e_idx]),
                    "duration_minutes": duration,
                    "peak_index": int(peak_idx),
                    "peak_time": pd.Timestamp(rec.ts[peak_idx]),
                    "contains_daily_solar_peak": bool(peak_idx == rec.daily_solar_peak_idx or (s_idx <= rec.daily_solar_peak_idx <= e_idx if rec.daily_solar_peak_idx is not None else False)),
                })
                cid += 1
    return rows


def generate_dense_candidates(cache: dict[tuple[str, str], DayRecord]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for rec in cache.values():
        rows.extend(generate_dense_candidates_for_record(rec))
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["candidate_id"] = out.groupby(["substation_id", "date"]).cumcount()
    return out


def assert_candidate_sanity(candidates: pd.DataFrame) -> None:
    null_counts = candidates.groupby(["substation_id", "date"])["is_null"].sum()
    bad_null = null_counts.loc[null_counts.ne(1)]
    if not bad_null.empty:
        raise AssertionError(f"Expected exactly one null candidate per site-day, found bad groups: {bad_null.head().to_dict()}")
    non = candidates.loc[~candidates["is_null"]].copy()
    if non.empty:
        return
    starts = pd.to_datetime(non["candidate_start"])
    ends = pd.to_datetime(non["candidate_end"])
    start_minutes = starts.dt.hour * 60 + starts.dt.minute
    end_minutes = ends.dt.hour * 60 + ends.dt.minute
    if not ((start_minutes >= WINDOW_START_HOUR * 60) & (end_minutes <= WINDOW_END_HOUR * 60)).all():
        raise AssertionError("Found non-null candidate outside the 06:00-18:00 bounds.")
    if not ((non["duration_minutes"] >= MIN_DURATION_MINUTES) & (non["duration_minutes"] <= MAX_DURATION_MINUTES)).all():
        raise AssertionError("Found non-null candidate outside duration bounds.")


## 5. Counterfactual Features And Relevance Labels

For every candidate window, this section builds lean v1 `m9.2_physics` features. The diagnostic label is a graded IoU relevance score, not a direct interval label:

- `4`: IoU at least 0.85;
- `3`: IoU at least 0.70;
- `2`: IoU at least 0.50;
- `1`: IoU at least 0.25;
- `0`: poor/no overlap.

On non-RPF days, the null candidate receives relevance `4`; all non-null candidates receive `0`. This lets the ranker learn to compare "best window" against "no RPF" inside the same site-day.


In [ ]:
def arr(values) -> np.ndarray:
    return np.asarray(values, dtype=float)


def finite_or_zero(value: float) -> float:
    return float(value) if np.isfinite(value) else 0.0


def nan_stat(values: np.ndarray, fn, default: float = 0.0) -> float:
    values = arr(values)
    if values.size == 0 or np.all(np.isnan(values)):
        return default
    try:
        return finite_or_zero(fn(values))
    except (ValueError, FloatingPointError):
        return default


def roughness(values: np.ndarray) -> float:
    values = arr(values)
    if values.size < 2:
        return 0.0
    return finite_or_zero(np.nansum(np.abs(np.diff(values))))


def curvature(values: np.ndarray) -> float:
    values = arr(values)
    if values.size < 3:
        return 0.0
    return finite_or_zero(np.nansum(np.abs(np.diff(values, n=2))))


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = arr(a)
    b = arr(b)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3 or np.nanstd(a[mask]) == 0 or np.nanstd(b[mask]) == 0:
        return 0.0
    return finite_or_zero(np.corrcoef(a[mask], b[mask])[0, 1])


def shape_score(values: np.ndarray, inverted: bool = False) -> float:
    values = arr(values)
    mask = np.isfinite(values)
    if mask.sum() < 4:
        return 0.0
    y = values.copy()
    if inverted:
        y = -y
    y = y[mask]
    span = np.nanmax(y) - np.nanmin(y)
    if not np.isfinite(span) or span <= 1e-9:
        return 0.0
    y_norm = (y - np.nanmin(y)) / span
    x = np.linspace(-1.0, 1.0, len(y_norm))
    bell = 1.0 - x**2
    bell = (bell - bell.min()) / (bell.max() - bell.min())
    return safe_corr(y_norm, bell)


def bridge_residual(values: np.ndarray, s: int, e: int) -> float:
    values = arr(values)
    if e < s or values.size == 0:
        return 0.0
    left_idx = max(s - 1, 0)
    right_idx = min(e + 1, len(values) - 1)
    left_val = values[left_idx]
    right_val = values[right_idx]
    seg = values[s : e + 1]
    if seg.size == 0 or not np.isfinite(left_val) or not np.isfinite(right_val):
        return 0.0
    line = np.linspace(left_val, right_val, len(seg))
    return finite_or_zero(np.nanmean(np.abs(seg - line)))


def boundary_jump(values: np.ndarray, s: int, e: int) -> float:
    values = arr(values)
    jump = 0.0
    if s > 0 and np.isfinite(values[s]) and np.isfinite(values[s - 1]):
        jump += abs(values[s] - values[s - 1])
    if e + 1 < len(values) and np.isfinite(values[e]) and np.isfinite(values[e + 1]):
        jump += abs(values[e + 1] - values[e])
    return finite_or_zero(jump)


def window_mask_from_times(rec: DayRecord, start: pd.Timestamp, end: pd.Timestamp) -> np.ndarray:
    ts = pd.to_datetime(rec.ts)
    return (ts >= pd.Timestamp(start)) & (ts <= pd.Timestamp(end))


def window_iou(rec: DayRecord, start: pd.Timestamp, end: pd.Timestamp) -> float:
    if not rec.has_rpf:
        return 0.0
    pred = window_mask_from_times(rec, start, end)
    true = rec.labels.astype(bool)
    union = np.logical_or(pred, true).sum()
    if union == 0:
        return 0.0
    return float(np.logical_and(pred, true).sum() / union)


def relevance_from_iou(iou: float) -> int:
    if iou >= 0.85:
        return 4
    if iou >= 0.70:
        return 3
    if iou >= 0.50:
        return 2
    if iou >= 0.25:
        return 1
    return 0


def candidate_index_bounds(rec: DayRecord, start: pd.Timestamp, end: pd.Timestamp) -> tuple[int, int] | None:
    mask = window_mask_from_times(rec, start, end)
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return None
    return int(idx[0]), int(idx[-1])


def build_features_for_candidate(rec: DayRecord, cand: pd.Series) -> dict[str, Any]:
    is_null = bool(cand["is_null"])
    base = {
        "dataset": rec.dataset,
        "substation_id": rec.substation_id,
        "date": rec.date,
        "candidate_id": int(cand["candidate_id"]),
        "is_null": is_null,
        "candidate_start": pd.NaT if is_null else pd.Timestamp(cand["candidate_start"]),
        "candidate_end": pd.NaT if is_null else pd.Timestamp(cand["candidate_end"]),
        "true_start": rec.true_start,
        "true_end": rec.true_end,
        "true_label_day": rec.has_rpf,
        "n_rows": len(rec.ts),
        "day_solar_peak": nan_stat(rec.solar, np.nanmax),
        "day_solar_p95": nan_stat(rec.solar, lambda x: np.nanpercentile(x, 95)),
        "day_net_p05": nan_stat(rec.net, lambda x: np.nanpercentile(x, 5)),
        "day_net_p95": nan_stat(rec.net, lambda x: np.nanpercentile(x, 95)),
        "day_missing_net": int(np.isnan(rec.net).sum()),
        "day_missing_solar": int(np.isnan(rec.solar).sum()),
    }
    if is_null:
        iou = 0.0
        relevance = 4 if not rec.has_rpf else 0
        base.update({
            "duration_minutes": 0.0,
            "start_hour": 0.0,
            "end_hour": 0.0,
            "mid_hour": 0.0,
            "month": int(pd.Timestamp(rec.date).month),
            "weekday": int(pd.Timestamp(rec.date).weekday()),
            "is_weekend": int(pd.Timestamp(rec.date).weekday() >= 5),
            "contains_daily_solar_peak": 0,
            "distance_to_daily_solar_peak_minutes": 999.0,
            "iou_with_true": iou,
            "relevance": relevance,
            "start_error_minutes": np.nan,
            "end_error_minutes": np.nan,
            "candidate_solar_peak": 0.0,
            "candidate_net_peak": 0.0,
            "candidate_pseudoload_std": 0.0,
            "candidate_pseudoload_range": 0.0,
            "candidate_pseudoload_roughness": 0.0,
            "counterfactual_roughness_delta": 0.0,
            "counterfactual_curvature_delta": 0.0,
            "bridge_residual_delta": 0.0,
            "boundary_jump_delta": 0.0,
            "solar_net_corr": 0.0,
            "derivative_same_sign_fraction": 0.0,
            "derivative_product_mean": 0.0,
            "ramp_up_same_sign": 0.0,
            "ramp_down_same_sign": 0.0,
            "solar_bell_score": 0.0,
            "net_n_shape_score": 0.0,
            "negative_reconstructed_fraction": 0.0,
        })
        return base

    s_e = candidate_index_bounds(rec, cand["candidate_start"], cand["candidate_end"])
    if s_e is None:
        raise ValueError("Non-null candidate has no matching timestamp interval.")
    s, e = s_e
    inside = slice(s, e + 1)
    solar = rec.solar
    net = rec.net
    u_empty = solar + net
    u_w = u_empty.copy()
    u_w[inside] = solar[inside] - net[inside]
    pseudo = solar[inside] - net[inside]
    solar_seg = solar[inside]
    net_seg = net[inside]
    dsolar = np.diff(solar_seg)
    dnet = np.diff(net_seg)
    same = np.sign(dsolar) == np.sign(dnet)
    valid_deriv = np.isfinite(dsolar) & np.isfinite(dnet)
    if valid_deriv.any():
        same_fraction = float(np.mean(same[valid_deriv]))
        derivative_product = float(np.nanmean(dsolar[valid_deriv] * dnet[valid_deriv]))
        ramp_up_mask = dsolar[valid_deriv] > 0
        ramp_down_mask = dsolar[valid_deriv] < 0
        ramp_up = float(np.mean(same[valid_deriv][ramp_up_mask])) if ramp_up_mask.any() else 0.0
        ramp_down = float(np.mean(same[valid_deriv][ramp_down_mask])) if ramp_down_mask.any() else 0.0
    else:
        same_fraction = derivative_product = ramp_up = ramp_down = 0.0

    duration = float(cand["duration_minutes"])
    start_ts = pd.Timestamp(cand["candidate_start"])
    end_ts = pd.Timestamp(cand["candidate_end"])
    mid_hour = (start_ts.hour + start_ts.minute / 60 + end_ts.hour + end_ts.minute / 60) / 2
    daily_peak_distance = 999.0
    if rec.daily_solar_peak_idx is not None:
        daily_peak_distance = min(
            abs((pd.Timestamp(rec.ts[rec.daily_solar_peak_idx]) - start_ts).total_seconds() / 60.0),
            abs((pd.Timestamp(rec.ts[rec.daily_solar_peak_idx]) - end_ts).total_seconds() / 60.0),
            0.0 if s <= rec.daily_solar_peak_idx <= e else 999.0,
        )
    iou = window_iou(rec, start_ts, end_ts)
    relevance = relevance_from_iou(iou) if rec.has_rpf else 0
    if rec.has_rpf:
        start_error = abs((start_ts - rec.true_start).total_seconds() / 60.0)
        end_error = abs((end_ts - rec.true_end).total_seconds() / 60.0)
    else:
        start_error = np.nan
        end_error = np.nan

    base.update({
        "duration_minutes": duration,
        "start_hour": start_ts.hour + start_ts.minute / 60.0,
        "end_hour": end_ts.hour + end_ts.minute / 60.0,
        "mid_hour": mid_hour,
        "month": int(start_ts.month),
        "weekday": int(start_ts.weekday()),
        "is_weekend": int(start_ts.weekday() >= 5),
        "contains_daily_solar_peak": int(bool(cand.get("contains_daily_solar_peak", False))),
        "distance_to_daily_solar_peak_minutes": finite_or_zero(daily_peak_distance),
        "iou_with_true": iou,
        "relevance": int(relevance),
        "start_error_minutes": start_error,
        "end_error_minutes": end_error,
        "candidate_solar_peak": nan_stat(solar_seg, np.nanmax),
        "candidate_net_peak": nan_stat(net_seg, np.nanmax),
        "candidate_net_p05": nan_stat(net_seg, lambda x: np.nanpercentile(x, 5)),
        "candidate_net_p95": nan_stat(net_seg, lambda x: np.nanpercentile(x, 95)),
        "candidate_solar_p95": nan_stat(solar_seg, lambda x: np.nanpercentile(x, 95)),
        "candidate_pseudoload_std": nan_stat(pseudo, np.nanstd),
        "candidate_pseudoload_range": nan_stat(pseudo, lambda x: np.nanmax(x) - np.nanmin(x)),
        "candidate_pseudoload_roughness": roughness(pseudo),
        "counterfactual_roughness_delta": roughness(u_empty) - roughness(u_w),
        "counterfactual_curvature_delta": curvature(u_empty) - curvature(u_w),
        "bridge_residual_delta": bridge_residual(u_empty, s, e) - bridge_residual(u_w, s, e),
        "boundary_jump_delta": boundary_jump(u_empty, s, e) - boundary_jump(u_w, s, e),
        "solar_net_corr": safe_corr(solar_seg, net_seg),
        "derivative_same_sign_fraction": finite_or_zero(same_fraction),
        "derivative_product_mean": finite_or_zero(derivative_product),
        "ramp_up_same_sign": finite_or_zero(ramp_up),
        "ramp_down_same_sign": finite_or_zero(ramp_down),
        "solar_bell_score": shape_score(solar_seg, inverted=False),
        "net_n_shape_score": shape_score(net_seg, inverted=False),
        "negative_reconstructed_fraction": float(np.mean((solar_seg - net_seg) < 0)) if len(solar_seg) else 0.0,
    })
    return base


def key_signature(keys: set[tuple[str, str]], dataset: str) -> str:
    payload = json.dumps({
        "dataset": dataset,
        "keys": sorted([list(k) for k in keys]),
        "feature_version": FEATURE_VERSION,
        "top_peaks": TOP_SOLAR_PEAKS,
        "run_mode": RUN_MODE,
    }, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


def build_or_load_feature_table(cache: dict[tuple[str, str], DayRecord], dataset: str) -> pd.DataFrame:
    sig = key_signature(set(cache.keys()), dataset)
    path = CACHE_DIR / f"{RUN_MODE}_{dataset}_{sig}_candidate_features.parquet"
    if REUSE_CACHED_FEATURES and path.exists():
        features = pd.read_parquet(path)
        print(f"Loaded cached {dataset} features: {path.name} ({len(features):,} rows)")
        return features
    candidates = generate_dense_candidates(cache)
    assert_candidate_sanity(candidates)
    rows: list[dict[str, Any]] = []
    grouped = candidates.groupby(["substation_id", "date"], sort=False)
    for key, group in grouped:
        rec = cache[key]
        for _, cand in group.iterrows():
            rows.append(build_features_for_candidate(rec, cand))
    features = pd.DataFrame(rows)
    features = features.replace([np.inf, -np.inf], np.nan)
    numeric = features.select_dtypes(include=[np.number]).columns
    features[numeric] = features[numeric].fillna(0.0)
    features.to_parquet(path, index=False)
    print(f"Wrote cached {dataset} features: {path.name} ({len(features):,} rows)")
    return features


alpha_features = build_or_load_feature_table(alpha_cache, "alpha")
beta_features = build_or_load_feature_table(beta_cache, "beta")

alpha_features.to_csv(CSV_DIR / f"{RUN_MODE}_01_alpha_candidate_features_preview.csv", index=False)
beta_features.to_csv(CSV_DIR / f"{RUN_MODE}_02_beta_candidate_features_preview.csv", index=False)

summary = (
    pd.concat([alpha_features, beta_features], ignore_index=True)
    .groupby(["dataset", "substation_id", "date"], as_index=False)
    .agg(
        candidate_count=("candidate_id", "size"),
        non_null_candidates=("is_null", lambda s: int((~s.astype(bool)).sum())),
        true_label_day=("true_label_day", "max"),
        best_iou=("iou_with_true", "max"),
        has_iou85=("iou_with_true", lambda s: bool((s >= 0.85).any())),
    )
)
summary.to_csv(CSV_DIR / f"{RUN_MODE}_03_candidate_summary.csv", index=False)
print(summary.groupby("dataset")[["candidate_count", "non_null_candidates", "best_iou"]].mean().round(3))


## 6. Ranking Model, Alpha Margin Selection, And Decoding

The ranker scores all candidates within a day, including the null candidate. Decoding compares the best non-null candidate against the null candidate. A positive day is emitted only when the non-null candidate beats null by the Alpha-selected margin.

This is the key structural difference from ordinary interval classification: the model is asked to choose the best event window, not independently label every timestamp.


In [ ]:
METADATA_COLUMNS = {
    "dataset",
    "substation_id",
    "date",
    "candidate_id",
    "candidate_start",
    "candidate_end",
    "true_start",
    "true_end",
    "true_label_day",
    "iou_with_true",
    "relevance",
    "start_error_minutes",
    "end_error_minutes",
}


def feature_columns(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        if col in METADATA_COLUMNS:
            continue
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            cols.append(col)
    return cols


FEATURE_COLUMNS = feature_columns(alpha_features)


def split_alpha_train_validation(features: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    keys = (
        features[["substation_id", "date", "true_label_day"]]
        .drop_duplicates()
        .sort_values(["true_label_day", "substation_id", "date"])
        .reset_index(drop=True)
    )
    train_keys = []
    val_keys = []
    for label, group in keys.groupby("true_label_day", sort=False):
        g = group.sort_values(["substation_id", "date"]).reset_index(drop=True)
        # Validation gets every fourth day, but never all days for a class.
        val_mask = (np.arange(len(g)) % 4 == 0) if len(g) >= 4 else np.zeros(len(g), dtype=bool)
        if val_mask.all():
            val_mask[-1] = False
        val_keys.extend(list(zip(g.loc[val_mask, "substation_id"], g.loc[val_mask, "date"])))
        train_keys.extend(list(zip(g.loc[~val_mask, "substation_id"], g.loc[~val_mask, "date"])))
    if not train_keys or not val_keys:
        shuffled = keys.sample(frac=1.0, random_state=RANDOM_SEED)
        cut = max(1, int(len(shuffled) * 0.75))
        train_keys = list(zip(shuffled.iloc[:cut]["substation_id"], shuffled.iloc[:cut]["date"]))
        val_keys = list(zip(shuffled.iloc[cut:]["substation_id"], shuffled.iloc[cut:]["date"]))
    train_set = set(train_keys)
    val_set = set(val_keys)
    temp = features.copy()
    temp["_key"] = list(zip(temp["substation_id"], temp["date"]))
    train = temp.loc[temp["_key"].isin(train_set)].drop(columns="_key").copy()
    val = temp.loc[temp["_key"].isin(val_set)].drop(columns="_key").copy()
    return train, val


def fit_candidate_model(train: pd.DataFrame, feature_cols: list[str]):
    train = train.sort_values(["substation_id", "date", "candidate_id"]).reset_index(drop=True)
    X = train[feature_cols].astype(float)
    y = train["relevance"].astype(float)
    groups = train.groupby(["substation_id", "date"], sort=False).size().to_numpy()
    try:
        model = xgb.XGBRanker(
            objective="rank:pairwise",
            eval_metric="ndcg",
            n_estimators=90 if RUN_MODE != "full" else 180,
            max_depth=3,
            learning_rate=0.06,
            subsample=0.90,
            colsample_bytree=0.90,
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=4,
        )
        model.fit(X, y, group=groups, verbose=False)
        return model, "xgb_ranker"
    except Exception as exc:
        print(f"XGBRanker failed, falling back to XGBClassifier: {exc}")
        y_bin = (train["relevance"] >= 3).astype(int)
        weights = np.where(train["relevance"] >= 3, 8.0, 1.0 + train["relevance"].astype(float))
        model = xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=90 if RUN_MODE != "full" else 180,
            max_depth=3,
            learning_rate=0.06,
            subsample=0.90,
            colsample_bytree=0.90,
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=4,
        )
        model.fit(train[feature_cols].astype(float), y_bin, sample_weight=weights, verbose=False)
        return model, "xgb_classifier_fallback"


def score_candidates(model, model_kind: str, features: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    out = features.copy()
    X = out[feature_cols].astype(float)
    if model_kind == "xgb_classifier_fallback":
        out["score"] = model.predict_proba(X)[:, 1]
    else:
        out["score"] = model.predict(X)
    return out


def decode_scored_days(scored: pd.DataFrame, margin_threshold: float, model_kind: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for (site, date), group in scored.groupby(["substation_id", "date"], sort=True):
        group = group.sort_values("candidate_id")
        null = group.loc[group["is_null"].astype(bool)]
        if len(null) != 1:
            raise AssertionError(f"{site} {date} expected one null candidate, found {len(null)}")
        null_row = null.iloc[0]
        non = group.loc[~group["is_null"].astype(bool)]
        if non.empty:
            best = None
            margin = -np.inf
            pred = False
        else:
            best = non.loc[non["score"].idxmax()]
            margin = float(best["score"] - null_row["score"])
            pred = bool(margin >= margin_threshold)
            if model_kind == "xgb_classifier_fallback":
                pred = bool(pred and best["score"] >= P_MIN)
        if not pred or best is None:
            rows.append({
                "substation_id": site,
                "date": date,
                "true_label_day": bool(null_row["true_label_day"]),
                "pred_label_day": False,
                "selected_candidate_id": np.nan,
                "pred_start": pd.NaT,
                "pred_end": pd.NaT,
                "selected_score": np.nan,
                "null_score": float(null_row["score"]),
                "score_margin": margin,
                "selected_iou": 0.0,
                "selected_relevance": 0,
            })
        else:
            rows.append({
                "substation_id": site,
                "date": date,
                "true_label_day": bool(best["true_label_day"]),
                "pred_label_day": True,
                "selected_candidate_id": int(best["candidate_id"]),
                "pred_start": best["candidate_start"],
                "pred_end": best["candidate_end"],
                "selected_score": float(best["score"]),
                "null_score": float(null_row["score"]),
                "score_margin": margin,
                "selected_iou": float(best["iou_with_true"]),
                "selected_relevance": int(best["relevance"]),
            })
    return pd.DataFrame(rows)


def binary_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)
    tp = int(np.logical_and(y_true, y_pred).sum())
    fp = int(np.logical_and(~y_true, y_pred).sum())
    fn = int(np.logical_and(y_true, ~y_pred).sum())
    tn = int(np.logical_and(~y_true, ~y_pred).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "support": int(len(y_true)),
        "positive_support": int(y_true.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def interval_metrics(decoded: pd.DataFrame, cache: dict[tuple[str, str], DayRecord]) -> dict[str, float]:
    y_true_all = []
    y_pred_all = []
    for _, row in decoded.iterrows():
        rec = cache[(row["substation_id"], row["date"])]
        pred = np.zeros(len(rec.ts), dtype=bool)
        if bool(row["pred_label_day"]) and pd.notna(row["pred_start"]) and pd.notna(row["pred_end"]):
            pred = window_mask_from_times(rec, row["pred_start"], row["pred_end"])
        y_true_all.append(rec.labels.astype(bool))
        y_pred_all.append(pred)
    if not y_true_all:
        return binary_metrics([], [])
    return binary_metrics(np.concatenate(y_true_all), np.concatenate(y_pred_all))


def day_metrics(decoded: pd.DataFrame) -> dict[str, float]:
    return binary_metrics(decoded["true_label_day"], decoded["pred_label_day"])


def sweep_margin_thresholds(scored_val: pd.DataFrame, model_kind: str) -> pd.DataFrame:
    raw = decode_scored_days(scored_val, margin_threshold=-np.inf, model_kind=model_kind)
    finite_margins = raw["score_margin"].replace([np.inf, -np.inf], np.nan).dropna()
    if finite_margins.empty:
        thresholds = np.array([0.0])
    else:
        low = float(np.nanpercentile(finite_margins, 5))
        high = float(np.nanpercentile(finite_margins, 95))
        thresholds = np.unique(np.r_[np.linspace(low, high, 41), 0.0])
    rows = []
    for threshold in thresholds:
        decoded = decode_scored_days(scored_val, margin_threshold=float(threshold), model_kind=model_kind)
        met = day_metrics(decoded)
        met["margin_threshold"] = float(threshold)
        rows.append(met)
    sweep = pd.DataFrame(rows).sort_values(["f1", "precision", "recall"], ascending=[False, False, False])
    return sweep.reset_index(drop=True)


alpha_train, alpha_val = split_alpha_train_validation(alpha_features)
print(f"Alpha train candidate rows: {len(alpha_train):,}; validation candidate rows: {len(alpha_val):,}")
model, model_kind = fit_candidate_model(alpha_train, FEATURE_COLUMNS)
alpha_val_scored = score_candidates(model, model_kind, alpha_val, FEATURE_COLUMNS)
margin_sweep = sweep_margin_thresholds(alpha_val_scored, model_kind)
selected_margin = float(margin_sweep.iloc[0]["margin_threshold"]) if not margin_sweep.empty else 0.0
alpha_val_decoded = decode_scored_days(alpha_val_scored, selected_margin, model_kind)

margin_sweep.to_csv(CSV_DIR / f"{RUN_MODE}_04_alpha_margin_sweep.csv", index=False)
alpha_val_decoded.to_csv(CSV_DIR / f"{RUN_MODE}_05_alpha_validation_decoded_days.csv", index=False)

print(f"Model kind: {model_kind}")
print(f"Selected Alpha margin threshold: {selected_margin:.4f}")
print("Alpha validation day metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in day_metrics(alpha_val_decoded).items()})


## 7. Beta Focus Evaluation

This section scores the selected Beta diagnostic days. These Beta results are exploratory only: they are useful for understanding whether the counterfactual candidate/ranker can recover hard examples such as `beta_B 2023-10-13`, but they are not publication-ready validation.


In [ ]:
beta_scored = score_candidates(model, model_kind, beta_features, FEATURE_COLUMNS)
beta_decoded = decode_scored_days(beta_scored, selected_margin, model_kind)

metrics_rows = []
for dataset_name, decoded, cache in [
    ("alpha_validation", alpha_val_decoded, alpha_cache),
    ("beta_focus" if RUN_MODE != "full" else "beta_transfer_full", beta_decoded, beta_cache),
]:
    dm = day_metrics(decoded)
    dm.update({"dataset": dataset_name, "level": "day", "model": MODEL_NAME, "run_mode": RUN_MODE})
    im = interval_metrics(decoded, cache)
    im.update({"dataset": dataset_name, "level": "interval", "model": MODEL_NAME, "run_mode": RUN_MODE})
    metrics_rows.extend([dm, im])
metrics = pd.DataFrame(metrics_rows)

beta_scored.to_csv(CSV_DIR / f"{RUN_MODE}_06_beta_scored_candidates.csv", index=False)
beta_decoded.to_csv(CSV_DIR / f"{RUN_MODE}_07_beta_decoded_days.csv", index=False)
metrics.to_csv(CSV_DIR / f"{RUN_MODE}_08_metrics.csv", index=False)

site_metrics_rows = []
for site, group in beta_decoded.groupby("substation_id", sort=True):
    dm = day_metrics(group)
    dm.update({"substation_id": site, "dataset": "beta_focus" if RUN_MODE != "full" else "beta_transfer_full", "level": "day", "model": MODEL_NAME, "run_mode": RUN_MODE})
    site_cache = {key: rec for key, rec in beta_cache.items() if key[0] == site}
    im = interval_metrics(group, site_cache)
    im.update({"substation_id": site, "dataset": "beta_focus" if RUN_MODE != "full" else "beta_transfer_full", "level": "interval", "model": MODEL_NAME, "run_mode": RUN_MODE})
    site_metrics_rows.extend([dm, im])
site_metrics = pd.DataFrame(site_metrics_rows)
site_metrics.to_csv(CSV_DIR / f"{RUN_MODE}_09_beta_site_metrics.csv", index=False)

print(metrics[["dataset", "level", "support", "positive_support", "precision", "recall", "f1"]].round(4).to_string(index=False))

bb = beta_features.loc[(beta_features["substation_id"].eq("beta_B")) & (beta_features["date"].eq("2023-10-13"))]
if not bb.empty:
    best_iou = float(bb["iou_with_true"].max())
    print(f"beta_B 2023-10-13 best dense-candidate IoU: {best_iou:.3f}")


## 8. Figures And Feature Importance

These figures are intentionally compact. They help answer the first implementation questions: did dense windows produce enough plausible candidates, did Alpha margin selection behave sensibly, and which lean physics features did the model use?


In [ ]:
def savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()


# Figure 1: Alpha margin sweep.
fig, ax = plt.subplots(figsize=(6.2, 3.6))
plot_sweep = margin_sweep.sort_values("margin_threshold")
ax.plot(plot_sweep["margin_threshold"], plot_sweep["precision"], label="Precision", color=JCOL["dark_blue"], linewidth=2)
ax.plot(plot_sweep["margin_threshold"], plot_sweep["recall"], label="Recall", color=JCOL["orange"], linewidth=2)
ax.plot(plot_sweep["margin_threshold"], plot_sweep["f1"], label="F1", color=JCOL["grey"], linewidth=2)
ax.axvline(selected_margin, color=JCOL["light_grey"], linestyle="--", linewidth=1.5, label="Selected")
ax.set_axisbelow(True)
ax.grid(True, axis="y", color=JCOL["light_white"], linewidth=0.8)
ax.set_xlabel("Non-null score minus null score")
ax.set_ylabel("Day-level score")
ax.set_ylim(0, 1.05)
ax.set_title("Alpha validation margin sweep")
ax.legend(frameon=False, ncol=2)
savefig(FIG_DIR / f"{RUN_MODE}_fig01_alpha_margin_sweep.png")


# Figure 2: candidate counts.
fig, ax = plt.subplots(figsize=(6.2, 3.6))
candidate_counts = summary.copy()
for dataset_name, color in [("alpha", JCOL["dark_blue"]), ("beta", JCOL["orange"])]:
    data = candidate_counts.loc[candidate_counts["dataset"].eq(dataset_name), "non_null_candidates"]
    if len(data):
        ax.hist(data, bins=16, alpha=0.68, label=dataset_name.title(), color=color)
ax.set_axisbelow(True)
ax.grid(True, axis="y", color=JCOL["light_white"], linewidth=0.8)
ax.set_xlabel("Non-null dense candidates per site-day")
ax.set_ylabel("Site-days")
ax.set_title("Dense candidate volume in current run")
ax.legend(frameon=False)
savefig(FIG_DIR / f"{RUN_MODE}_fig02_candidate_counts.png")


# Figure 3: metrics.
fig, ax = plt.subplots(figsize=(6.4, 3.8))
metric_plot = metrics.loc[metrics["level"].eq("day")].copy()
x = np.arange(len(metric_plot))
width = 0.22
for offset, metric_name, color in [(-width, "precision", JCOL["dark_blue"]), (0, "recall", JCOL["orange"]), (width, "f1", JCOL["grey"])]:
    ax.bar(x + offset, metric_plot[metric_name], width, label=metric_name.title(), color=color)
ax.set_xticks(x)
ax.set_xticklabels(metric_plot["dataset"], rotation=0)
ax.set_ylim(0, 1.05)
ax.set_axisbelow(True)
ax.grid(True, axis="y", color=JCOL["light_white"], linewidth=0.8)
ax.set_ylabel("Day-level score")
ax.set_title("m9.2_physics smoke/focus day metrics")
ax.legend(frameon=False, ncol=3)
savefig(FIG_DIR / f"{RUN_MODE}_fig03_day_metrics.png")


# Figure 4: feature importance, if available.
importance = pd.DataFrame()
if hasattr(model, "feature_importances_"):
    importance = pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": model.feature_importances_})
    importance = importance.sort_values("importance", ascending=False)
    importance.to_csv(CSV_DIR / f"{RUN_MODE}_10_feature_importance.csv", index=False)
    top = importance.head(16).sort_values("importance")
    fig, ax = plt.subplots(figsize=(6.8, 5.0))
    ax.barh(top["feature"], top["importance"], color=JCOL["orange"])
    ax.set_axisbelow(True)
    ax.grid(True, axis="x", color=JCOL["light_white"], linewidth=0.8)
    ax.set_xlabel("XGBoost importance")
    ax.set_title("Top m9.2_physics features")
    savefig(FIG_DIR / f"{RUN_MODE}_fig04_feature_importance.png")
else:
    pd.DataFrame({"feature": FEATURE_COLUMNS}).to_csv(CSV_DIR / f"{RUN_MODE}_10_feature_importance.csv", index=False)

print(f"Wrote figures to {FIG_DIR}")


## 9. Plotly HTML Examples

Each HTML example overlays raw net load, solar generation, the no-RPF reconstruction `U_empty`, the selected-window reconstruction `U_W`, the manual window, and the selected m9.2 window. These are the fastest way to see whether the model is selecting physically plausible windows.


In [ ]:
def selected_reconstruction(rec: DayRecord, start, end) -> np.ndarray:
    u = rec.solar + rec.net
    if pd.notna(start) and pd.notna(end):
        mask = window_mask_from_times(rec, start, end)
        u = u.copy()
        u[mask] = rec.solar[mask] - rec.net[mask]
    return u


def write_example_html(rec: DayRecord, decoded_row: pd.Series, scored_group: pd.DataFrame, out_path: Path) -> None:
    ts = pd.to_datetime(rec.ts)
    u_empty = rec.solar + rec.net
    u_selected = selected_reconstruction(rec, decoded_row["pred_start"], decoded_row["pred_end"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts, y=rec.net, mode="lines", name="Raw net load", line=dict(color=JCOL["dark_blue"], width=2)))
    fig.add_trace(go.Scatter(x=ts, y=rec.solar, mode="lines", name="Solar", line=dict(color=JCOL["orange"], width=2)))
    fig.add_trace(go.Scatter(x=ts, y=u_empty, mode="lines", name="U_empty = solar + net", line=dict(color=JCOL["light_grey"], width=1.5, dash="dot")))
    fig.add_trace(go.Scatter(x=ts, y=u_selected, mode="lines", name="Selected U_W", line=dict(color=JCOL["grey"], width=2, dash="dash")))

    if rec.has_rpf:
        fig.add_vrect(x0=rec.true_start, x1=rec.true_end, fillcolor="rgba(92,125,153,0.20)", line_width=0, annotation_text="Manual", annotation_position="top left")
    if bool(decoded_row["pred_label_day"]) and pd.notna(decoded_row["pred_start"]) and pd.notna(decoded_row["pred_end"]):
        fig.add_vrect(x0=decoded_row["pred_start"], x1=decoded_row["pred_end"], fillcolor="rgba(235,147,44,0.22)", line_width=1, line_color=JCOL["orange"], annotation_text="m9.2", annotation_position="top right")

    top = scored_group.loc[~scored_group["is_null"].astype(bool)].sort_values("score", ascending=False).head(3)
    for _, cand in top.iterrows():
        fig.add_vrect(
            x0=cand["candidate_start"],
            x1=cand["candidate_end"],
            fillcolor="rgba(47,77,103,0.06)",
            line_width=1,
            line_color="rgba(47,77,103,0.25)",
        )

    title = (
        f"{MODEL_NAME} {rec.substation_id} {rec.date} | "
        f"pred={bool(decoded_row['pred_label_day'])} true={rec.has_rpf} "
        f"margin={decoded_row['score_margin']:.3f} IoU={decoded_row['selected_iou']:.3f}"
    )
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1000,
        height=560,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        xaxis_title="Timestamp",
        yaxis_title="MW",
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(out_path)


# Clear stale examples for this run mode so old partial runs cannot be confused with current output.
for stale_html in HTML_DIR.glob(f"{RUN_MODE}_*.html"):
    stale_html.unlink()

html_rows = []
if WRITE_HTML:
    # Keep examples deterministic and small.
    scored_lookup = {key: group.copy() for key, group in beta_scored.groupby(["substation_id", "date"], sort=False)}
    for _, row in beta_decoded.sort_values(["substation_id", "date"]).head(18).iterrows():
        key = (row["substation_id"], row["date"])
        rec = beta_cache[key]
        confusion = (
            "TP" if row["true_label_day"] and row["pred_label_day"]
            else "FN" if row["true_label_day"] and not row["pred_label_day"]
            else "FP" if (not row["true_label_day"]) and row["pred_label_day"]
            else "TN"
        )
        out_name = f"{RUN_MODE}_{confusion}_{key[0]}_{key[1]}__{MODEL_NAME.replace('.', '_')}.html"
        out_path = HTML_DIR / out_name
        write_example_html(rec, row, scored_lookup[key], out_path)
        html_rows.append({
            "substation_id": key[0],
            "date": key[1],
            "confusion": confusion,
            "true_label_day": bool(row["true_label_day"]),
            "pred_label_day": bool(row["pred_label_day"]),
            "score_margin": float(row["score_margin"]),
            "selected_iou": float(row["selected_iou"]),
            "html_file": str(out_path.relative_to(OUTPUT_ROOT)),
        })
html_index = pd.DataFrame(html_rows)
html_index.to_csv(CSV_DIR / f"{RUN_MODE}_11_html_index.csv", index=False)
print(f"Wrote {len(html_index)} HTML examples")


## 10. Manifest And Readiness Notes

The manifest records exactly what was run. Smoke/focus artifacts are diagnostic only. If this approach looks promising, the next step is a deliberate full Alpha LOSO run with locked controls and a separate Beta transfer evaluation.


In [ ]:
manifest = {
    "model_name": MODEL_NAME,
    "run_mode": RUN_MODE,
    "publication_ready": False,
    "exploratory_warning": "Beta focus labels are used only for diagnostics/evaluation; smoke/focus metrics are not publication-ready validation.",
    "full_mode_executed": RUN_MODE == "full",
    "feature_version": FEATURE_VERSION,
    "model_kind": model_kind,
    "selected_alpha_margin_threshold": selected_margin,
    "p_min_classifier_fallback": P_MIN,
    "controls": {
        "window_start_hour": WINDOW_START_HOUR,
        "window_end_hour": WINDOW_END_HOUR,
        "min_duration_minutes": MIN_DURATION_MINUTES,
        "max_duration_minutes": MAX_DURATION_MINUTES,
        "top_solar_peaks": TOP_SOLAR_PEAKS,
        "solar_peak_min_frac": SOLAR_PEAK_MIN_FRAC,
        "solar_peak_min_separation_minutes": SOLAR_PEAK_MIN_SEPARATION_MINUTES,
        "reuse_cached_features": REUSE_CACHED_FEATURES,
        "write_html": WRITE_HTML,
    },
    "row_counts": {
        "alpha_feature_rows": int(len(alpha_features)),
        "beta_feature_rows": int(len(beta_features)),
        "alpha_train_rows": int(len(alpha_train)),
        "alpha_validation_rows": int(len(alpha_val)),
        "beta_decoded_days": int(len(beta_decoded)),
    },
    "metrics": metrics.to_dict(orient="records"),
    "outputs": {
        "csv_dir": str(CSV_DIR),
        "figure_dir": str(FIG_DIR),
        "html_dir": str(HTML_DIR),
        "cache_dir": str(CACHE_DIR),
    },
}
manifest_path = MANIFEST_DIR / f"{RUN_MODE}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print(f"Wrote manifest: {manifest_path}")

print("\nKey outputs:")
for path in [
    CSV_DIR / f"{RUN_MODE}_03_candidate_summary.csv",
    CSV_DIR / f"{RUN_MODE}_08_metrics.csv",
    CSV_DIR / f"{RUN_MODE}_09_beta_site_metrics.csv",
    FIG_DIR / f"{RUN_MODE}_fig01_alpha_margin_sweep.png",
    FIG_DIR / f"{RUN_MODE}_fig03_day_metrics.png",
    manifest_path,
]:
    print(f"- {path}")
